In [ ]:
# --- Load python regression modules ---

# numpy: Fundamental library for numerical operations and array manipulation.
import numpy as np

# pandas: Core data manipulation library used for dataframes and data cleaning.
import pandas as pd

# LinearRegression: Ordinary Least Squares (OLS) algorithm that fits a straight line/hyperplane.
from sklearn.linear_model import LinearRegression

# Evaluation metrics:
# - mean_squared_error: Penalizes larger prediction errors heavily (squared scale).
# - r2_score: Coefficient of Determination; fraction of target variance explained by features.
from sklearn.metrics import mean_squared_error, r2_score

# train_test_split: Randomly partitions data to prevent data leakage and evaluate generalization.
from sklearn.model_selection import train_test_split

# KNeighborsRegressor: Non-parametric algorithm that predicts outcomes by averaging nearest data points.
from sklearn.neighbors import KNeighborsRegressor

In [ ]:
# --- Import the genders, heights, and weights dataset ---

# Read CSV file into a pandas DataFrame (tabular structure of rows and columns).
df = pd.read_csv('sample_data/genders_heights_weights.csv')

In [ ]:
# --- Check the dimensions of the dataset ---

# .shape -> returns a tuple (total_rows, total_columns)
print(f"Dataset Shape: {df.shape}")

# .ndim  -> returns the number of axes (2 for a tabular 2D DataFrame)
print(f"Dimensions (ndim): {df.ndim}")

# .size  -> returns the total number of cells (rows multiplied by columns)
print(f"Total Elements (size): {df.size}\n")

Dataset Shape: (10000, 3)
Dimensions (ndim): 2
Total Elements (size): 30000



In [ ]:
# --- Examine the first few data samples ---

# .head() displays the top 5 records by default to inspect column formatting and sample values.
print(df.head(), "\n")

  Gender  Height  Weight
0   Male   187.6   109.7
1   Male   174.7    73.6
2   Male   188.2    96.5
3   Male   182.2    99.8
4   Male   177.5    93.6 



In [ ]:
# --- Randomly sample 10 data points ---

# .sample(10) selects rows uniformly at random to check for data anomalies across the entire distribution.
ten_dp = df.sample(10)
print(ten_dp)

      Gender  Height  Weight
4254    Male   170.8    82.9
3703    Male   172.5    83.2
4607    Male   179.1    93.3
8503  Female   159.5    57.1
441     Male   179.9    89.4
4146    Male   168.2    75.3
5613  Female   157.6    61.5
8330  Female   159.8    47.2
4505    Male   174.0    87.4
3161    Male   164.6    71.2


In [ ]:
# -- Describe the statistical summary of the dataset ---

# include='all' computes descriptive stats for both numeric (mean, std, min, quartiles, max)
# and categorical columns (unique count, top occurring value, frequency).
stastical_summary = df.describe(include='all')
print("--- Statistical Summary ---")
print(stastical_summary, "\n")

--- Statistical Summary ---
       Gender        Height        Weight
count   10000  10000.000000  10000.000000
unique      2           NaN           NaN
top      Male           NaN           NaN
freq     5000           NaN           NaN
mean      NaN    168.573940     73.228260
std       NaN      9.772842     14.563851
min       NaN    137.800000     29.300000
25%       NaN    161.300000     61.600000
50%       NaN    168.400000     73.100000
75%       NaN    175.700000     84.900000
max       NaN    200.700000    122.500000 



In [ ]:
# --- Isolate the target variable from the features ---

# Establish a single-feature baseline first (Height -> Weight).
# Double brackets [['Height']] -> keeps X as a 2D DataFrame (required by scikit-learn).
X_base = df[['Height']]

# Single brackets ['Weight'] -> extracts y as a 1D pandas Series (target vector).
y = df['Weight']

In [ ]:
# --- Split data into training and testing sets ---

# test_size=0.2: 80% used to train the model, 20% reserved strictly for testing.
# random_state=42: Fixes pseudo-random seed so splits remain reproducible across runs.
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_base, y, test_size=0.2, random_state=42
)

In [ ]:
# --- Verify the dimensions of both split sets ---

# Verify row counts match expectations (e.g., 80/20 proportion) and column counts match feature sets.
print(f"Baseline Train shape: X={X_train_b.shape}, y={y_train_b.shape}")
print(f"Baseline Test shape:  X={X_test_b.shape}, y={y_test_b.shape}\n")

Baseline Train shape: X=(8000, 1), y=(8000,)
Baseline Test shape:  X=(2000, 1), y=(2000,)



In [ ]:
#  --- Initialize and train a Linear Regression model ---

# Instantiate the estimator object.
base_lr = LinearRegression()

# .fit() calculates optimal model coefficients (slope and intercept) via Ordinary Least Squares.
base_lr.fit(X_train_b, y_train_b)

LinearRegression()

In [ ]:
# --- Evaluate the initial baseline R² score ---

# Generate predictions using unseen test features.
y_pred_base = base_lr.predict(X_test_b)

# r2_score measures goodness of fit (1.0 is a perfect fit, 0.0 equals a horizontal mean baseline).
r2_base = r2_score(y_test_b, y_pred_base)
print(f"Baseline Linear Regression R² Score (Height only): {r2_base:.4f}\n")

Baseline Linear Regression R² Score (Height only): 0.8577



In [ ]:
# --- Apply one-hot encoding to the Gender column ---

# Linear models cannot ingest raw strings like 'Male' or 'Female'.
# drop_first=True prevents multicollinearity (the "dummy variable trap") by creating
# a single binary column (e.g., Gender_Male: 1 for Male, 0 for Female).
df_encoded = pd.get_dummies(df, columns=['Gender'], drop_first=True)

In [ ]:
# --- Re-isolate the expanded features and target variable ---

# Drop 'Weight' to leave only predictive feature columns (Height, Gender_Male).
X_enc = df_encoded.drop(columns=['Weight'])

# Isolate target variable again.
y_enc = df_encoded['Weight']

In [ ]:
# --- Re-split the newly encoded dataset ---

# Use the same random_state=42 to ensure identical train/test index splits as baseline.
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_enc, y_enc, test_size=0.2, random_state=42
)

In [ ]:
# --- Train a new Linear Regression model ---

lr_encoded = LinearRegression()

# Model now learns two coefficients: beta_1 for Height and beta_2 for Gender indicator.
lr_encoded.fit(X_train_e, y_train_e)

LinearRegression()

In [ ]:
# --- Compare the improved R² performance score ---

y_pred_e = lr_encoded.predict(X_test_e)
r2_enc = r2_score(y_test_e, y_pred_e)

print(f"Improved Linear Regression R² Score (Height + Gender): {r2_enc:.4f}")
# Measure the exact quantitative gain achieved by adding the gender feature.
print(f"R² Improvement: {r2_enc - r2_base:+.4f}\n")

Improved Linear Regression R² Score (Height + Gender): 0.9033
R² Improvement: +0.0456



In [ ]:
# --- Train k-NN Regressor and loop through k values to optimize ---

print("--- Optimizing k-NN Regressor ---")

# Evaluate odd k values from 1 to 25 to find the optimal bias-variance tradeoff:
# - Low k (e.g., k=1): High variance / overfits training noise.
# - High k: High bias / oversmooths and underfits underlying patterns.
k_values = range(1, 26, 2)
best_k = None
best_r2 = -float('inf')  # Initialize to negative infinity to ensure first score beats it

for k in k_values:
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_train_e, y_train_e)

    # In scikit-learn regressors, .score(X, y) returns the R² metric directly.
    score = knn.score(X_test_e, y_test_e)
    print(f"k = {k:2d} | R² Score: {score:.4f}")

    # Track best hyperparameter configuration
    if score > best_r2:
        best_r2 = score
        best_k = k

print(f"\nOptimal k: {best_k} with R² Score: {best_r2:.4f}\n")

# Re-fit a final k-NN model instance using the discovered optimal k.
optimal_knn = KNeighborsRegressor(n_neighbors=best_k)
optimal_knn.fit(X_train_e, y_train_e)

--- Optimizing k-NN Regressor ---
k =  1 | R² Score: 0.8111
k =  3 | R² Score: 0.8723
k =  5 | R² Score: 0.8836
k =  7 | R² Score: 0.8882
k =  9 | R² Score: 0.8908
k = 11 | R² Score: 0.8920
k = 13 | R² Score: 0.8933
k = 15 | R² Score: 0.8951
k = 17 | R² Score: 0.8954
k = 19 | R² Score: 0.8965
k = 21 | R² Score: 0.8975
k = 23 | R² Score: 0.8984
k = 25 | R² Score: 0.8990

Optimal k: 25 with R² Score: 0.8990



KNeighborsRegressor(n_neighbors=25)

In [ ]:
# --- Create an interactive custom user-input testing section ---

print("--- Custom Prediction Tool ---")

# Dynamically locate the encoded gender column name (e.g., 'Gender_Male')
# so the script adapts automatically regardless of naming conventions.
gender_col_name = [col for col in X_enc.columns if 'Gender' in col][0]

try:
    # 1. Collect inputs from the user
    user_gender = input("Enter Gender (Male/Female): ").strip().capitalize()
    user_height_cm = float(input("Enter Height (cm): "))

    # 2. Extract the active category label (e.g., 'Male' from 'Gender_Male')
    target_category = gender_col_name.split('_')[-1]

    # Assign 1 if user matches the encoded dummy column, otherwise 0 for reference group
    gender_val = 1 if user_gender == target_category else 0

    # 3. Assemble inputs into a DataFrame matching exact column order and names of X_enc
    custom_data = pd.DataFrame([[user_height_cm, gender_val]], columns=X_enc.columns)

    # 4. Predict target values with both trained models
    pred_lr = lr_encoded.predict(custom_data)[0]
    pred_knn = optimal_knn.predict(custom_data)[0]

    # 5. Output comparative predictions
    print("\nPredicted Weight:")
    print(f"- Linear Regression : {pred_lr:.2f}")
    print(f"- k-NN (k={best_k})       : {pred_knn:.2f}")

except ValueError:
    # Handles non-numeric inputs for height gracefully without crashing the script
    print("Invalid input entered. Please provide valid numeric height values.")

--- Custom Prediction Tool ---
Enter Gender (Male/Female): female
Enter Height (cm): 162

Predicted Weight:
- Linear Regression : 61.81
- k-NN (k=25)       : 62.36
